In [1]:
# ================================
# 🎤 ChatGPT por voz (Colab)
# ================================

# 📦 Instalar dependências
!pip install -q git+https://github.com/openai/whisper.git openai gTTS

# ================================
# 🔐 API KEY (segura)
# ================================
import os, getpass
os.environ['OPENAI_API_KEY'] = getpass.getpass("Digite sua API Key: ")

import openai
openai.api_key = os.environ.get('OPENAI_API_KEY')

# ================================
# 📚 Imports
# ================================
import whisper
from gtts import gTTS
from IPython.display import Audio, display, Javascript
from google.colab import output
from base64 import b64decode

language = 'pt'

# ================================
# 🎤 GRAVAÇÃO DE ÁUDIO
# ================================
RECORD = """
const sleep  = time => new Promise(resolve => setTimeout(resolve, time))
const b2text = blob => new Promise(resolve => {
  const reader = new FileReader()
  reader.onloadend = e => resolve(e.srcElement.result)
  reader.readAsDataURL(blob)
})
var record = time => new Promise(async resolve => {
  stream = await navigator.mediaDevices.getUserMedia({ audio: true })
  recorder = new MediaRecorder(stream)
  chunks = []
  recorder.ondataavailable = e => chunks.push(e.data)
  recorder.start()
  await sleep(time)
  recorder.onstop = async ()=>{
    blob = new Blob(chunks)
    text = await b2text(blob)
    resolve(text)
  }
  recorder.stop()
})
"""

def gravar_audio(segundos=5):
    display(Javascript(RECORD))
    js_result = output.eval_js(f'record({segundos * 1000})')
    audio = b64decode(js_result.split(',')[1])

    file_name = '/content/audio.wav'
    with open(file_name, 'wb') as f:
        f.write(audio)

    return file_name

# ================================
# 🧠 TRANSCRIÇÃO (Whisper)
# ================================
model = whisper.load_model("base")

def transcrever_audio(arquivo):
    result = model.transcribe(arquivo, fp16=False, language=language)
    return result["text"]

# ================================
# 💬 CHATGPT
# ================================
def perguntar_chatgpt(messages):
    response = openai.ChatCompletion.create(
        model="gpt-4o-mini",
        messages=messages
    )
    return response.choices[0].message.content

# ================================
# 🔊 TEXTO → VOZ
# ================================
def falar(texto):
    tts = gTTS(text=texto, lang=language)
    arquivo = "/content/resposta.mp3"
    tts.save(arquivo)
    display(Audio(arquivo, autoplay=True))

# ================================
# 🔁 LOOP PRINCIPAL (ASSISTENTE)
# ================================
messages = []

print("🤖 Assistente iniciado! Diga 'sair' para encerrar.\n")

while True:
    try:
        print("🎤 Ouvindo...")

        audio_file = gravar_audio(5)
        display(Audio(audio_file))

        texto = transcrever_audio(audio_file)
        print("🧑 Você:", texto)

        if not texto.strip():
            print("⚠️ Não entendi, tente novamente.\n")
            continue

        if "sair" in texto.lower():
            print("👋 Encerrando...")
            break

        messages.append({"role": "user", "content": texto})

        resposta = perguntar_chatgpt(messages)
        print("🤖 ChatGPT:", resposta)

        messages.append({"role": "assistant", "content": resposta})

        falar(resposta)

    except Exception as e:
        print("❌ Erro:", e)
        print("Tentando novamente...\n")

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.3/188.3 MB 6.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
typer 0.24.1 requires click>=8.2.1, but you have click 8.1.8 which is incompatible.
Digite sua API Key: ··········


100%|████████████████████████████████████████| 139M/139M [00:00<00:00, 229MiB/s]


🤖 Assistente iniciado! Diga 'sair' para encerrar.

🎤 Ouvindo...


<IPython.core.display.Javascript object>

KeyboardInterrupt: 